<a href="https://colab.research.google.com/github/aiportfoliorhea/ai-portfolio/blob/main/loan_bot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn chromadb langchain pypdf sentence-transformers langchain-community langchain-text-splitters langchain-chroma anthropic

In [ ]:
import chromadb
import langchain
import fastapi

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="loanbot",
    embedding_function=embeddings,
)

print("Vector store ready")

In [ ]:
# 2. Create a simple test file to work with

sample_text = """
LOAN AGREEMENT

Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.

Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.

Section 3: Late Payment
Payments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.

Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.
"""

# Save it as a text file
with open("sample_loan.txt", "w") as f:
    f.write(sample_text)

print("Sample loan doc created")

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample_loan.txt")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")

In [ ]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {chunks[i]}")

In [ ]:
vector_store.add_documents(chunks)
print("Chunks stored in vector store")

In [ ]:
retriever = vector_store.as_retriever()
results = retriever.invoke("what is the loan amount?")
print(results)

In [ ]:
for doc in results:
    print(doc.page_content)
    print("---")

In [ ]:
import anthropic
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"]  = userdata.get('anthropic_key')
print("fetched Anthropic key succesfully")

In [ ]:
from anthropic import Anthropic

client = Anthropic()

def ask_loanbot(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""Act as a legal loan document assistant. Answer the question using only the context provided.
If the answer is not in the context, say "I don't have that information in the document. Don't make up any anwers, strictly stick to the context given to you."

Context:
{context}

Question: {question}"""
            }
        ]
    )

    return message.content[0].text

print(ask_loanbot("what is the loan amount?"))
print("----------- ----------- -----------")
print(ask_loanbot("what is the weather in Dallas?"))
